# Exploração dos dados

Pequena exploração dos dados para ter um melhor entendimento do dataset

### Load the data and first cleaning

In [5]:
import pandas as pd
import numpy as np

#load files
df_teams = pd.read_csv("hockey_development_data/Teams.csv")
df_scoring = pd.read_csv("hockey_development_data/Scoring.csv")
df_goalies = pd.read_csv("hockey_development_data/Goalies.csv")
df_coaches = pd.read_csv("hockey_development_data/Coaches.csv")

tables = [df_teams, df_scoring, df_goalies, df_coaches]

# clean spaces
for df in tables:
    df.columns = df.columns.str.strip()

# remove black spaces
def clean_strings(df):
    df_obj = df.select_dtypes(['object'])
    df[df_obj.columns] = df_obj.apply(lambda x: x.str.strip())
    return df

# clean and filter above 1990
for i in range(len(tables)):
    tables[i] = clean_strings(tables[i])
    tables[i] = tables[i][(tables[i]['lgID'] == 'NHL') & (tables[i]['year'] >= 1990)].copy()
    tables[i]['year'] = tables[i]['year'].astype(int)

df_teams, df_scoring, df_goalies, df_coaches = tables

# print("Teams")
# display(df_teams.sort_values(by=['year', 'tmID']).tail())

# print("\nCoaches")
# display(df_coaches.sort_values(by=['year', 'tmID']).head())

# print("\nScoring")
# display(df_scoring.sort_values(by=['year', 'tmID']).head())

# print("\nGoalies")
# display(df_goalies.sort_values(by=['year', 'tmID']).head())

C:\Users\cosme\AppData\Local\Temp\ipykernel_26180\2382758702.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_obj = df.select_dtypes(['object'])
C:\Users\cosme\AppData\Local\Temp\ipykernel_26180\2382758702.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-str

### Group individual stats to use in the team

In [6]:
#convert all in numbers (0 if null)
cols_scoring = ['G', 'A', 'Pts', 'PIM']
df_scoring[cols_scoring] = df_scoring[cols_scoring].apply(pd.to_numeric).fillna(0)

#group by year and sum the statistics
teams_stats = df_scoring.groupby(['tmID', 'year']).agg(
    teams_goals = ('G', 'sum'),       
    teams_pts = ('Pts', 'sum'),       
    teams_pim = ('PIM', 'sum')        
).reset_index()

#same but to the goalies
cols_goalies = ['SA', 'GA', 'Min']
df_goalies[cols_goalies] = df_goalies[cols_goalies].apply(pd.to_numeric).fillna(0)

goalie_stats = df_goalies.groupby(['tmID', 'year']).agg(
    team_SA = ('SA', 'sum'),
    team_GA = ('GA', 'sum')
).reset_index()

#Save percentage of the team
goalie_stats['team_save_pct'] = np.where(
    goalie_stats['team_SA'] > 0, 
    1 - (goalie_stats['team_GA'] / goalie_stats['team_SA']), 
    0
)

print("Teams stats")
display(teams_stats.head())

print("\nGoalie stats")
display(goalie_stats.head())

Teams stats


,tmID,year,teams_goals,teams_pts,teams_pim
0,ANA,1993,229.0,588.0,1483.0
1,ANA,1994,125.0,336.0,715.0
2,ANA,1995,234.0,637.0,1685.0
3,ANA,1996,245.0,662.0,1688.0
4,ANA,1997,205.0,523.0,1833.0



Goalie stats


,tmID,year,team_SA,team_GA,team_save_pct
0,ANA,1993,2606.0,241.0,0.907521
1,ANA,1994,1580.0,158.0,0.900000
2,ANA,1995,2634.0,242.0,0.908125
3,ANA,1996,2682.0,227.0,0.915362
4,ANA,1997,2483.0,252.0,0.898510


### Group by team

The players are part of the team, so their info should also be added in the team info

In [7]:
df_master = pd.merge(df_teams, teams_stats, on = ['tmID', 'year'], how = 'left')
df_master = pd.merge(df_master, goalie_stats, on = ['tmID', 'year'], how = 'left')

#since some years had less games we will use per game metric and not per season
df_master['G'] = df_master['G'].replace(0, 1) #cant divide by 0
df_master['win_pct'] = df_master['W'] / df_master['G']
df_master['teams_pts_per_game'] = df_master['teams_pts'] / df_master['G']

display(df_master[['year', 'tmID', 'W', 'win_pct', 'teams_pts', 'teams_pts_per_game']].head())

,year,tmID,W,win_pct,teams_pts,teams_pts_per_game
0,1990,BOS,44,0.5500,779.0,9.7375
1,1990,BUF,31,0.3875,750.0,9.3750
2,1990,CAL,46,0.5750,927.0,11.5875
3,1990,CHI,49,0.6125,756.0,9.4500
4,1990,DET,34,0.4250,742.0,9.2750


### Block the info about the Y year prediction

Since to train the data to predict the Y year champion we will need to use the info about Y-1 years available #NEEDS TO BE CHECKED

In [8]:
targets = df_master[['franchID', 'year', 'rank']].copy()

#remove rank so we cant "cheat"
features = df_master.drop(columns=['rank']).copy()
features['target_year'] = features['year'] + 1

#2004 we didnt had games
features.loc[features['year'] == 2003, 'target_year'] = 2005

df_model_ready = pd.merge(
    targets, 
    features, 
    left_on=['franchID', 'year'], 
    right_on=['franchID', 'target_year'], 
    suffixes=('_target', '_feature')
)

df_model_ready = df_model_ready.rename(columns={'year_target': 'target_season', 'year_feature': 'historical_season'})
df_model_ready = df_model_ready.drop(columns=['target_year'])

display(df_model_ready[['franchID', 'target_season', 'rank', 'historical_season', 'win_pct', 'teams_pts_per_game']].head())

,franchID,target_season,rank,historical_season,win_pct,teams_pts_per_game
0,BOS,1991,2,1990,0.5500,9.7375
1,BUF,1991,3,1990,0.3875,9.3750
2,CAL,1991,5,1990,0.5750,11.5875
3,CHI,1991,2,1990,0.6125,9.4500
4,DET,1991,1,1990,0.4250,9.2750


### Curate player-level statistics

Remove birth/death date and city fields, set position-inapplicable stats to `NaN`, and add skater/goalie efficiency metrics. The team-level features are saved separately because the player-season source has no team ID.

In [9]:
player_stats_source = "hockey_development_data/player_stats.csv"
player_stats_output = "hockey_development_data/player_stats_curated.csv"

df_player_stats = pd.read_csv(player_stats_source)
df_player_stats.columns = df_player_stats.columns.str.strip()

demographic_date_columns = {
    "birthYear", "birthMon", "birthDay",
    "deathYear", "deathMon", "deathDay",
}
columns_to_drop = [
    column for column in df_player_stats.columns
    if column in demographic_date_columns or column.lower().endswith("city")
]
df_player_stats = df_player_stats.drop(columns=columns_to_drop)

if "pos" not in df_player_stats.columns:
    raise KeyError("The player stats data must contain a 'pos' column.")

df_player_stats["pos"] = df_player_stats["pos"].astype("string").str.strip().str.upper()
df_player_stats["pos"] = df_player_stats["pos"].replace("", pd.NA)
position_known = df_player_stats["pos"].notna()
goalie_rows = df_player_stats["pos"].eq("G")
skater_rows = position_known & ~goalie_rows

skater_stat_columns = [
    column for column in df_player_stats.columns
    if column.startswith("skater_")
]
goalie_stat_columns = [
    column for column in df_player_stats.columns
    if column.startswith("goalie_")
]

df_player_stats.loc[goalie_rows | ~position_known, skater_stat_columns] = np.nan
df_player_stats.loc[skater_rows | ~position_known, goalie_stat_columns] = np.nan

if "is_goalie" in df_player_stats.columns:
    df_player_stats["is_goalie"] = pd.Series(pd.NA, index=df_player_stats.index, dtype="boolean")
    df_player_stats.loc[position_known, "is_goalie"] = goalie_rows.loc[position_known]

assert not any(
    column in demographic_date_columns or column.lower().endswith("city")
    for column in df_player_stats.columns
)
assert df_player_stats.loc[goalie_rows, skater_stat_columns].isna().all().all()
assert df_player_stats.loc[skater_rows, goalie_stat_columns].isna().all().all()
assert df_player_stats.loc[~position_known, skater_stat_columns + goalie_stat_columns].isna().all().all()

df_player_stats.to_csv(player_stats_output, index=False)
print(f"Curated {len(df_player_stats):,} player-season rows.")
print(f"Removed columns: {columns_to_drop}")
print(f"Output: {player_stats_output}")
display(df_player_stats.head())

C:\Users\cosme\AppData\Local\Temp\ipykernel_26180\1535515280.py:4: DtypeWarning: Columns (68: nameNote) have mixed types. Specify dtype option on import or set low_memory=False.
  df_player_stats = pd.read_csv(player_stats_source)


Curated 17,092 player-season rows.
Removed columns: ['birthYear', 'birthMon', 'birthDay', 'birthCity', 'deathYear', 'deathMon', 'deathDay', 'deathCity']
Output: hockey_development_data/player_stats_curated.csv


,playerID,year,skater_GP,skater_G,skater_A,skater_Pts,skater_PIM,skater_+/-,skater_PPG,skater_PPA,...,firstNHL,lastNHL,firstWHA,lastWHA,pos,birthCountry,birthState,deathCountry,deathState,is_goalie
0,aaltoan01,1997,3.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,...,1997.0,2000.0,NaN,NaN,C,Finland,NaN,NaN,NaN,False
1,aaltoan01,1998,73.0,3.0,5.0,8.0,24.0,-12.0,2.0,1.0,...,1997.0,2000.0,NaN,NaN,C,Finland,NaN,NaN,NaN,False
2,aaltoan01,1999,63.0,7.0,11.0,18.0,26.0,-13.0,1.0,0.0,...,1997.0,2000.0,NaN,NaN,C,Finland,NaN,NaN,NaN,False
3,aaltoan01,2000,12.0,1.0,1.0,2.0,2.0,1.0,0.0,0.0,...,1997.0,2000.0,NaN,NaN,C,Finland,NaN,NaN,NaN,False
4,abdelju01,2007,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,2007.0,2009.0,NaN,NaN,L,USA,MI,NaN,NaN,False


### Derive team-season model features

Run this cell to regenerate `player_stats_curated.csv` with individual metrics and `team_stats_curated.csv` with the six team-season features.

In [10]:
from runpy import run_path

run_path("hockey_development_data/curate_player_stats.py", run_name="__main__")

Player-season rows: 17,092
Team-season rows: 1,265
Player output: hockey_development_data\player_stats_curated.csv
Team output: hockey_development_data\team_stats_curated.csv


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'hockey_development_data/curate_player_stats.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, locals=None, fromlist=(), l